# Stage 1: 多来源数据读入 + obs schema 标准化

通过 `read_with_manifest` 读入原始单细胞数据并自动完成 13 个标准化步骤
（详见 SPEC "read_with_manifest"），包括基线 QC 指标计算，写入 stage 1 checkpoint h5ad。

**为什么这个 notebook 只做数据读入？** 多来源数据的格式、obs 列名、基因 ID 体系各异。
把标准化集中在 stage 1，后续所有分析直接操作统一的 AnnData，无需每步检查"这个数据集
的列名是什么"。

**本 notebook 产出**：
- `obs` 列跨来源数据集标准化（Layer 1 核心 + Layer 2 CellxGene 对齐 + Layer 3 项目自定义）
- 基线 QC 指标（`n_genes`, `total_counts`, `pct_counts_mt`, `pct_counts_ribo`）基于原始 counts 计算
- `var.index` = gene symbol + `var["ensembl_id"]` = Ensembl ID
- Stage 1 checkpoint `.h5ad` 文件，供 stage 2 QC 使用

In [ ]:
# === PARAMS ===
# 运行前编辑:
#   MANIFEST_PATH —— 读哪个数据集
#   OUTPUT_PATH   —— stage 1 checkpoint 写到哪里
#   RANDOM_SEED   —— 固定随机种子保证可复现
# 注意: 更换 MANIFEST_PATH 时记得同步更新 OUTPUT_PATH

MANIFEST_PATH = "data/nancang/manifest.yaml"   # 数据集 manifest 路径
OUTPUT_PATH   = "results/nancang_stage1_loaded_v1.h5ad"
RANDOM_SEED   = 42

In [ ]:
# 确保框架 src/ 在 sys.path 上，CWD 为项目根目录。
# 自动检测两种运行场景：从 notebooks/（Jupyter）还是项目根目录（nbconvert）启动。
import sys, os
_root = os.getcwd()
if not os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
    _root = os.path.abspath(os.path.join(_root, ".."))
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
print(f"PROJECT_ROOT: {_root}")


In [ ]:
# 导入（scanpy 原生 API + 框架函数仅在真正有缺口时使用）
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import yaml
import warnings

from scrna_integration import read_with_manifest

# 抑制 mygene / scanpy 在读取过程中的 chatty 警告
warnings.filterwarnings("ignore", message=".*mygene.*")
warnings.filterwarnings("ignore", message=".*Layer2.*")

sc.settings.verbosity = 2  # 显示有用进度（0=quiet, 3=verbose）
import importlib.metadata; print(f"Scanpy {importlib.metadata.version('scanpy')}  |  anndata {importlib.metadata.version('anndata')}")

In [ ]:
# 调用 read_with_manifest——框架唯一的 IO 入口。
# 内部自动完成 13 个标准化步骤（见 SPEC "read_with_manifest"），返回普通 AnnData。
print(f"\n===== 正在从 {MANIFEST_PATH} 读入 =====\n")
adata = read_with_manifest(MANIFEST_PATH)

print(f"\n返回 AnnData: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")

In [ ]:
# Quick visual check: obs head, var head, and uns keys.
# PI / students inspect these to verify the manifest-driven schema is correct
# before proceeding to stage 2 QC.
print("=== obs.head() ===")
display(adata.obs.head())

print("\n=== var.head() ===")
display(adata.var.head())

print("\n=== obs columns ===")
print(list(adata.obs.columns))

print("\n=== uns keys ===")
print(list(adata.uns.keys()))

print(f"\n=== obs.dtypes sample ===")
print(adata.obs.dtypes.head(10))

In [ ]:
# 基线 QC 指标——read_with_manifest 第 12 步自动计算。
# 这些是原始 counts 上的逐细胞指标，跨所有来源数据集对齐——
# 无论上游是否已做预处理都保证存在且列名一致。
print("===== 基线 QC 摘要（原始 counts）=====")
for col in ["n_genes", "total_counts", "pct_counts_mt", "pct_counts_ribo"]:
    if col in adata.obs.columns:
        vals = adata.obs[col]
        print(f"  {col:20s}: 均值={vals.mean():.1f}  中位={vals.median():.1f}  最小={vals.min():.1f}  最大={vals.max():.1f}")
print(f"\n  source_dataset: {sorted(adata.obs['source_dataset'].unique())}")
n_samples = adata.obs['sample_id'].nunique() if "sample_id" in adata.obs.columns else "N/A"
print(f"  n_samples:         {n_samples}")

# 同时检查 raw_matrix_path（stage 2 SoupX 需要）
rp = adata.uns.get("raw_matrix_path", None)
print(f"\n  raw_matrix_path（供 SoupX 用）: {rp}")

In [ ]:
# 内存纪律自检（写入前单行断言——SPEC Memory Discipline）。
# 守护最高风险的内存退化：adata.X 变为稠密或丢失 float32。
# 如果断言失败，追溯是哪步操作 densify 或 cast 了矩阵。
import scipy.sparse as sp
import numpy as np
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 不变量违反: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("内存自检通过: X 是 sparse CSR float32。")

In [ ]:
# 写入 stage checkpoint 到磁盘。
# compression="lzf" 是 Memory Discipline #4——比 gzip 快，
# 比未压缩约省 30% 空间，且保留 sparse CSR 布局。
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"已写入 {OUTPUT_PATH}")

# 校验文件已写入且可读。
import os
assert os.path.exists(OUTPUT_PATH), f"输出未找到: {OUTPUT_PATH}"
print(f"已校验: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")

In [ ]:
# 跨 stage 边界释放内存（Memory Discipline #3）。
# 没有这段的话，Jupyter 内核会保留上一 stage 的 AnnData，
# 在同一内核会话中运行下一 stage 时造成内存叠加。
del adata
import gc
gc.collect()
print("内存已释放。")